Setup (imports and basic paths)

In [6]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path

# Define paths
project_root = Path.cwd().parent  
data_dir = project_root / 'data'
out_dir = data_dir / '03_nlp'
out_dir.mkdir(parents=True, exist_ok=True)

# Sample window (386 months, 1990-01 to 2022-02)
date_range = pd.date_range('1990-01-01', '2022-02-01', freq='MS')
print(f"Sample size: {len(date_range)} months")
print(f"Output directory: {out_dir}")

Sample size: 386 months
Output directory: c:\Users\HP\Desktop\replication+contribution\data\03_nlp


In [7]:
def align_to_sample(df):
    """Ensure DataFrame index is month-start and matches our 386-month window."""
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df.index = df.index.to_period('M').to_timestamp()
    return df.reindex(date_range)

GPR Index

In [8]:
# Caldara-Iacoviello GPR (global geopolitical risk)
file_path = out_dir / 'gpr_monthly.csv'
gpr_url = 'https://www.matteoiacoviello.com/gpr_files/gpr_web_latest.xlsx'

if file_path.exists():
    df_gpr = pd.read_csv(file_path, index_col=0, parse_dates=True)
    print("Loaded GPR from cache.")
else:
    print("Downloading GPR index...")
    r = requests.get(gpr_url, timeout=30)
    if r.status_code != 200:
        raise Exception(f"Download failed, HTTP {r.status_code}")
    
    # Read the second sheet (index 1) because first sheet is README
    xl = pd.ExcelFile(r.content)
    print("Sheet names:", xl.sheet_names)
    df_raw = xl.parse(xl.sheet_names[1])
    print("Original column names:", list(df_raw.columns))
    
    # Keep only needed columns: first one (date) and those containing 'GPR'
    # The sheet usually has columns: 'month', 'GPR', 'GPRT', 'GPRA', etc.
    # We'll find them by name (case-insensitive)
    date_col = df_raw.columns[0]
    df_raw['date'] = pd.to_datetime(df_raw[date_col], errors='coerce')
    df_raw = df_raw.dropna(subset=['date']).set_index('date')
    
    # Look for GPR, GPRT, GPRA (case-insensitive)
    gpr_col = None
    threats_col = None
    acts_col = None
    for col in df_raw.columns:
        col_upper = col.upper()
        if col_upper == 'GPR':
            gpr_col = col
        elif col_upper in ['GPRT', 'GPR_THREAT', 'GPR_THREATS']:
            threats_col = col
        elif col_upper in ['GPRA', 'GPR_ACT', 'GPR_ACTS']:
            acts_col = col
    
    if gpr_col is None:
        # Fallback: take the first numeric column after date as GPR
        gpr_col = df_raw.columns[0]
        print(f"Warning: no column named 'GPR' found. Using '{gpr_col}' as GPR.")
    
    # Build DataFrame
    df_gpr = pd.DataFrame(index=df_raw.index)
    df_gpr['gpr'] = df_raw[gpr_col]
    if threats_col:
        df_gpr['gpr_threats'] = df_raw[threats_col]
    else:
        print("Warning: no threats column found. Setting to NaN.")
        df_gpr['gpr_threats'] = np.nan
    if acts_col:
        df_gpr['gpr_acts'] = df_raw[acts_col]
    else:
        print("Warning: no acts column found. Setting to NaN.")
        df_gpr['gpr_acts'] = np.nan
    
    # Align to sample window and save
    df_gpr = align_to_sample(df_gpr)
    df_gpr.to_csv(file_path)
    print(f"Saved GPR to {file_path}")

print(f"GPR coverage: {df_gpr['gpr'].notna().sum()}/386 months")
print(df_gpr.describe().round(2))

Loaded GPR from cache.
GPR coverage: 381/386 months
          gpr  gpr_threats  gpr_acts
count  381.00       381.00    381.00
mean    89.34        92.70     72.15
std     66.61        73.68     59.79
min     23.74        20.27     13.26
25%     47.05        46.70     39.47
50%     67.28        68.53     56.35
75%    110.27       116.79     81.97
max    545.26       602.37    497.12


 EA-GPR Index

In [9]:
# Bondarenko EA-GPR (euro area perspective)
ea_path = out_dir / 'ea_gpr_monthly.csv'
ea_url = 'https://github.com/YvesSchueler/EuroAreaGPR/raw/refs/heads/main/EA_GPR_Paper.xlsx'

if ea_path.exists():
    df_ea = pd.read_csv(ea_path, index_col=0, parse_dates=True)
    print("Loaded EA-GPR from cache.")
else:
    print("Downloading EA-GPR...")
    r = requests.get(ea_url, timeout=30)
    if r.status_code != 200:
        raise Exception(f"Download failed, HTTP {r.status_code}")
    
    xl = pd.ExcelFile(r.content)
    df_raw = xl.parse('MonthlyData')
    df_raw.columns = [c.strip() for c in df_raw.columns]
    
    # First column is date
    df_raw['date'] = pd.to_datetime(df_raw.iloc[:, 0], errors='coerce')
    df_raw = df_raw.dropna(subset=['date']).set_index('date')
    
    # Extract EA GPR column (named 'EA GPR' in the spreadsheet)
    ea_col = [c for c in df_raw.columns if 'EA GPR' in c][0]
    df_ea = df_raw[[ea_col]].copy()
    df_ea.columns = ['ea_gpr']
    df_ea['ea_gpr_diff'] = df_ea['ea_gpr'].diff()
    
    # Align and save
    df_ea = align_to_sample(df_ea)
    df_ea.to_csv(ea_path)
    print(f"Saved EA-GPR to {ea_path}")

print(f"EA-GPR coverage: {df_ea['ea_gpr'].notna().sum()}/386 months")
print(df_ea.describe().round(3))

Loaded EA-GPR from cache.
EA-GPR coverage: 290/386 months
        ea_gpr  ea_gpr_diff
count  290.000      289.000
mean     1.224        0.006
std      0.333        0.262
min      0.787       -1.070
25%      1.039       -0.121
50%      1.143       -0.001
75%      1.302        0.111
max      3.620        1.268


### WUI – World Uncertainty Index (Quarterly → Monthly)

**Source:**  
[World Uncertainty Index (WUI) – Quarterly Data](https://www.policyuncertainty.com/wui_quarterly.html)  
Ahir, H, N Bloom, and D Furceri (2022). *The World Uncertainty Index*, NBER Working Paper.  
The data are constructed by counting the frequency of the word “uncertain” (and its variants) in the Economist Intelligence Unit country reports, then scaling by total words (×1,000,000).  
A value of 12,496 means roughly 1.25% of words in those reports conveyed uncertainty.

**Our sample:**  
- Original frequency: quarterly (1990Q1 – 2022Q1)  
- We forward‑fill each quarterly value to the three subsequent months to obtain a monthly series.  
- The resulting WUI column is aligned to our 386‑month window (1990‑01 to 2022‑02).

**Coverage:** 386/386 months.  
No gaps, full sample coverage.  

The CSV file `wui.csv` (placed in `data/03_nlp/`) was manually downloaded from the above link (the “Excel” button for the global GDP‑weighted index). The cell below reads and processes it.

WUI – World Uncertainty Index 

In [10]:
# WUI – World Uncertainty Index (official quarterly CSV)
wui_path = out_dir / 'wui.csv'

if wui_path.exists():
    print("Loading WUI from local CSV...")
    # Read the CSV, skip metadata header rows
    df_wui_raw = pd.read_csv(
        wui_path,
        sep=";",
        skiprows=3,
        usecols=[0, 2],
        names=["quarter_str", "wui"],
        encoding="utf-8",
        engine="python"
    )
    # Clean quarter strings and convert to date (end of quarter)
    df_wui_raw["quarter_str"] = (
        df_wui_raw["quarter_str"]
        .str.replace(" ", "")
        .str.lower()
        .str.strip()
    )
    df_wui_raw["date"] = pd.PeriodIndex(
        df_wui_raw["quarter_str"], freq="Q"
    ).to_timestamp(how="end")
    df_wui_raw = df_wui_raw.dropna(subset=["date"]).set_index("date")
    
    # --- THIS IS THE FIX: remove the thousands separator (space) ---
    df_wui_raw["wui"] = (
        df_wui_raw["wui"]
        .astype(str)
        .str.replace(" ", "")          # remove space
        .str.replace("\xa0", "")        # just in case there's a non-breaking space
    )
    df_wui_raw["wui"] = pd.to_numeric(df_wui_raw["wui"], errors="coerce")
    print("First 4 rows of quarterly data:")
    print(df_wui_raw.head(4))
    
    # Forward-fill to monthly
    monthly_idx = pd.date_range("1990-01-01", "2022-02-01", freq="MS")
    df_wui = df_wui_raw.reindex(monthly_idx, method="ffill")
    
    # Back-fill the first two months (Jan, Feb 1990) if they are NaN
    df_wui = df_wui.fillna(method="bfill")
    
    # Align to exact sample window
    df_wui = align_to_sample(df_wui)
    df_wui = df_wui[["wui"]]   # keep only the wui column
    
    # Save clean monthly version
    df_wui.to_csv(out_dir / "wui_monthly.csv")
    print(f"Saved monthly WUI to {out_dir / 'wui_monthly.csv'}")
else:
    print(f"WUI file not found at {wui_path}. Please place wui.csv in the correct directory.")
    df_wui = pd.DataFrame(index=date_range, columns=["wui"])

print(f"WUI coverage: {df_wui['wui'].notna().sum()}/386 months")
print(df_wui.describe().round(2))

Loading WUI from local CSV...
First 4 rows of quarterly data:
                              quarter_str    wui
date                                            
1990-03-31 23:59:59.999999999      1990q1  12496
1990-06-30 23:59:59.999999999      1990q2   8770
1990-09-30 23:59:59.999999999      1990q3  16199
1990-12-31 23:59:59.999999999      1990q4  11872
Saved monthly WUI to c:\Users\HP\Desktop\replication+contribution\data\03_nlp\wui_monthly.csv
WUI coverage: 386/386 months
            wui
count    386.00
mean   17400.84
std     9004.05
min     5570.00
25%    11553.00
50%    14211.00
75%    21509.00
max    55685.00


C:\Users\HP\AppData\Local\Temp\ipykernel_5104\123004505.py:44: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_wui = df_wui.fillna(method="bfill")


### GDELT US–China Dyadic Events
**Source:** [GDELT 1.0 Event Database](https://www.gdeltproject.org/data.html#eventdata) – daily zipped CSV files, 1979‑present.  
**Why:** Directly measures the frequency and nature of bilateral interactions between the US and China.  
- **Event counts:** protests (CAMEO 14,15), military posturing (CAMEO 19,20), diplomatic visits (CAMEO 03,04,05,06) – monthly.  
- **Sentiment proxy:** average Goldstein scale per event (‑10 to +10).  
- **Theme proportions:** counts of all EventRootCode quad‑classes 01‑20, forming a 20‑column monthly matrix that acts as a structured analogue of topic‑model proportions.  

**Note:** The original plan requests NLP sentiment on news headlines. Because a 30‑year full‑text news corpus is not freely available, we use the GDELT Goldstein score – a per‑event sentiment rating generated from the same news reports – which closely tracks the tone of US‑China relations.

In [11]:
import os
import zipfile
import io
import time
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from datetime import datetime, timedelta

out_dir = Path.cwd().parent / "data" / "03_nlp"
out_dir.mkdir(parents=True, exist_ok=True)

# GDELT 1.0 events URL template
BASE_URL = "http://data.gdeltproject.org/events/{date}.export.CSV.zip"

# Target dyad: USA <-> CHN
US_CODE = "USA"
CHN_CODE = "CHN"

# CAMEO root codes for event types of interest
PROTEST_CODES = {14, 15}
MILITARY_CODES = {19, 20}
DIPLOMATIC_CODES = {3, 4, 5, 6}   # root codes 03-06: visits, consultations

# The GDELT 1.0 header (simplified, after unzipping the CSV)
# Column indices: 0=GLOBALEVENTID, 5=Actor1Code, 15=Actor2Code,
# 26=EventRootCode, 30=GoldsteinScale, etc.
ACTOR1_COL = 5
ACTOR2_COL = 15
EVENTROOT_COL = 26
GOLDSTEIN_COL = 30

def process_date(date_str):
    """
    Download and parse one daily GDELT event file.
    Returns a DataFrame of US-CHN dyadic events with needed columns, or None if error.
    """
    url = BASE_URL.format(date=date_str)
    try:
        resp = requests.get(url, timeout=60)
        if resp.status_code != 200:
            return None
    except Exception:
        return None
    
    try:
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            # The CSV inside has the same name as the zip but with .CSV
            csv_name = date_str + ".export.CSV"
            with zf.open(csv_name) as f:
                df = pd.read_csv(f, sep='\t', header=None, dtype=str, low_memory=False)
    except Exception:
        return None

    # Filter to rows where Actor1 is USA/CHN and Actor2 is CHN/USA (either direction)
    mask_us_chn = (
        ((df[ACTOR1_COL] == US_CODE) & (df[ACTOR2_COL] == CHN_CODE)) |
        ((df[ACTOR1_COL] == CHN_CODE) & (df[ACTOR2_COL] == US_CODE))
    )
    df_dyad = df[mask_us_chn].copy()
    if df_dyad.empty:
        return None

    # Extract needed fields
    df_dyad['EventRootCode'] = pd.to_numeric(df_dyad[EVENTROOT_COL], errors='coerce')
    df_dyad['GoldsteinScale'] = pd.to_numeric(df_dyad[GOLDSTEIN_COL], errors='coerce')
    df_dyad = df_dyad.dropna(subset=['EventRootCode'])

    # Add binary indicators for event types
    df_dyad['protest'] = df_dyad['EventRootCode'].isin(PROTEST_CODES).astype(int)
    df_dyad['military'] = df_dyad['EventRootCode'].isin(MILITARY_CODES).astype(int)
    df_dyad['diplomatic'] = df_dyad['EventRootCode'].isin(DIPLOMATIC_CODES).astype(int)

    # Quad‑class (01-04 → 1, 05-08 → 2, …) – we'll just keep raw root code for later aggregation
    df_dyad['root_class'] = ((df_dyad['EventRootCode'] - 1) // 4 + 1).astype(int)  # 1..5

    return df_dyad[['EventRootCode', 'GoldsteinScale', 'protest', 'military',
                     'diplomatic', 'root_class']]

In [12]:
# Date range: 1990-01-01 to 2022-02-28 (month end)
start_date = datetime(1990, 1, 1)
end_date = datetime(2022, 2, 28)

# Cache directory to avoid re-downloading processed days
cache_dir = out_dir / "gdel_daily_cache"
cache_dir.mkdir(exist_ok=True)

daily_counts = []
current_date = start_date
while current_date <= end_date:
    date_str = current_date.strftime("%Y%m%d")
    cache_file = cache_dir / f"{date_str}.parquet"
    
    if cache_file.exists():
        # Already processed – skip download
        current_date += timedelta(days=1)
        continue
    
    df_day = process_date(date_str)
    if df_day is not None and not df_day.empty:
        # Add a date column for aggregation
        df_day['date'] = current_date
        daily_counts.append(df_day)
        # Cache the day's data
        df_day.to_parquet(cache_file)
    
    # Print progress every 100 days
    if current_date.day % 100 == 1:
        print(f"Processed {current_date.date()}")
    
    current_date += timedelta(days=1)
    time.sleep(0.05)  # polite scraping

print("Done downloading/processing all days.")

# Combine all cached files into one large DataFrame
all_days = []
for f in cache_dir.glob("*.parquet"):
    all_days.append(pd.read_parquet(f))
if all_days:
    df_all_events = pd.concat(all_days, ignore_index=True)
    df_all_events.to_parquet(out_dir / "gdel_uschn_events_raw.parquet")
    print(f"Saved {len(df_all_events)} events to raw.parquet")
else:
    df_all_events = pd.DataFrame()
    print("No events found.")

Processed 1990-01-01


KeyboardInterrupt: 

In [ ]:
# ── Monthly aggregation directly from the raw Parquet ─────────────────────────
raw_path = out_dir / "gdel_uschn_events_raw.parquet"

if raw_path.exists():
    df = pd.read_parquet(raw_path)
    print(f"Loaded raw events: {len(df)} rows")
    print(f"Columns: {list(df.columns)}")

    # Ensure date is datetime and set as index
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])
    df = df.set_index('date')

    # Resample to month-start and aggregate
    monthly = df.resample('MS').agg(
        protest_count      = ('protest', 'sum'),
        military_count     = ('military', 'sum'),
        diplomatic_count   = ('diplomatic', 'sum'),
        total_events       = ('protest', 'size'),
        goldstein_mean     = ('GoldsteinScale', 'mean'),
        goldstein_sum      = ('GoldsteinScale', 'sum')
    )

    # Theme matrix (20 quad-classes)
    theme_dummies = pd.get_dummies(df['root_class'], prefix='theme')
    theme_monthly = theme_dummies.resample('MS').sum()
    all_themes = [f'theme_{c}' for c in range(1, 21)]
    theme_monthly = theme_monthly.reindex(columns=all_themes, fill_value=0)

    # Combine and align to sample window
    df_monthly = pd.concat([monthly, theme_monthly], axis=1)
    date_range = pd.date_range('1990-01-01', '2022-02-01', freq='MS')
    df_monthly = df_monthly.reindex(date_range)

    # Save
    out_csv = out_dir / "gdel_events_monthly.csv"
    df_monthly.to_csv(out_csv)
    print(f"Monthly features saved → {out_csv}")
    print(f"Shape: {df_monthly.shape}")
    display(df_monthly.describe().round(2))
else:
    print(f"Raw Parquet file not found: {raw_path}")

Loaded raw events: 1211111 rows
Columns: ['EventRootCode', 'GoldsteinScale', 'protest', 'military', 'diplomatic', 'root_class', 'date']
Monthly features saved → c:\Users\HP\Desktop\replication+contribution\data\03_nlp\gdel_events_monthly.csv
Shape: (386, 26)


,protest_count,military_count,diplomatic_count,total_events,goldstein_mean,goldstein_sum,theme_1,theme_2,theme_3,theme_4,...,theme_11,theme_12,theme_13,theme_14,theme_15,theme_16,theme_17,theme_18,theme_19,theme_20
count,102.00,102.00,102.0,102.00,102.00,102.00,102.0,102.0,102.00,102.00,...,102.00,102.00,102.00,102.00,102.00,102.00,102.0,102.00,102.00,102.00
mean,124.98,605.04,0.0,11873.64,1.22,14370.10,0.0,0.0,1002.09,391.36,...,2014.82,559.95,757.71,37.23,383.53,515.61,0.0,242.84,42.14,106.84
std,56.53,216.56,0.0,3574.91,0.46,6918.66,0.0,0.0,299.39,154.55,...,1001.68,247.87,228.22,28.03,146.47,327.11,0.0,114.27,25.56,47.04
min,42.00,281.00,0.0,5917.00,-0.35,-6374.30,0.0,0.0,515.00,143.00,...,816.00,174.00,392.00,5.00,151.00,170.00,0.0,77.00,11.00,15.00
25%,87.00,450.00,0.0,8804.50,0.87,9729.92,0.0,0.0,791.75,282.25,...,1488.50,379.25,590.00,19.00,282.00,306.00,0.0,185.50,27.50,71.00
50%,111.00,564.50,0.0,11826.50,1.30,14917.20,0.0,0.0,963.50,350.50,...,1884.50,520.00,713.50,29.00,352.50,426.00,0.0,225.50,36.00,101.50
75%,156.00,720.00,0.0,14018.50,1.54,18137.38,0.0,0.0,1229.50,483.25,...,2258.75,702.25,904.25,48.50,459.75,579.25,0.0,274.25,50.00,132.00
max,330.00,1269.00,0.0,23451.00,2.07,36541.70,0.0,0.0,1804.00,910.00,...,8598.00,1393.00,1526.00,197.00,941.00,1811.00,0.0,1023.00,175.00,286.00


In [3]:
# Inspect the raw Parquet that was just saved
raw_path = out_dir / "gdel_uschn_events_raw.parquet"
df_raw = pd.read_parquet(raw_path)
print("Columns:", list(df_raw.columns))
print("First 3 rows:")
print(df_raw.head(3))

Columns: ['EventRootCode', 'GoldsteinScale', 'protest', 'military', 'diplomatic', 'root_class', 'date']
First 3 rows:
   EventRootCode  GoldsteinScale  protest  military  diplomatic  root_class  \
0             13             0.4        0         0           0           4   
1             10             0.0        0         0           0           3   
2             10             0.0        0         0           0           3   

        date  
0 2013-09-01  
1 2013-09-01  
2 2013-09-01  


### GDELT US–China Dyadic Events (full window, via `gdelt` package)

**Source:** [GDELT 1.0 Event Database](https://www.gdeltproject.org/data.html#eventdata) accessed via the official [`gdelt` Python package](https://pypi.org/project/gdelt/) (version 0.1.14).  
**Coverage:** GDELT 1.0 archives reach back to **1 January 1979** and are updated daily. The package handles both the recent daily files and the older yearly archives transparently.

**Why this method:**  
- Uses the same `gdelt` library you already installed.  
- Calls `gdelt.gdelt(version=1)` to target the 1.0 dataset, which is the only one with full historical coverage.  
- Queries year‑by‑year with `coverage=True` to ensure every single day is downloaded (even if it takes a few hours — you said time is not an issue).  
- Caches each year as a Parquet file so a second run is instant.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

out_dir = Path.cwd().parent / "data" / "03_nlp"
out_dir.mkdir(parents=True, exist_ok=True)

# Cache folder for yearly raw data
cache_dir = out_dir / "gdel_yearly_cache_v1"
cache_dir.mkdir(exist_ok=True)

# ── Instantiate GDELT 1.0 ──────────────────────────────────────────────────────
from gdelt import gdelt
gd1 = gdelt(version=1)   # correct instantiation   # Version 1 = full historical coverage (1979‑present)

# ── Constants ───────────────────────────────────────────────────────────────────
US_CODE  = "USA"
CHN_CODE = "CHN"

PROTEST_CODES    = {14, 15}
MILITARY_CODES   = {19, 20}
DIPLOMATIC_CODES = {3, 4, 5, 6}

def fetch_year_v1(year):
    """
    Fetch all US‑CHN events for a single year using GDELT 1.0.
    Uses coverage=True so every daily file is requested.
    Returns a cleaned DataFrame or empty.
    """
    start = f"{year}-01-01"
    end   = f"{year}-12-31"
    try:
        # coverage=True makes the library fetch every single day
        df = gd1.Search([start, end],
                         table='events',
                         output='pd',
                         coverage=True)
    except Exception as e:
        print(f"   Error {year}: {e}")
        return pd.DataFrame()

    if df is None or df.empty:
        return pd.DataFrame()

    # The package returns columns named after the GDELT 1.0 table. 
    # We need: 'SQLDATE', 'Actor1Code', 'Actor2Code', 'EventRootCode', 'GoldsteinScale'
    needed = ['SQLDATE', 'Actor1Code', 'Actor2Code', 'EventRootCode', 'GoldsteinScale']
    for col in needed:
        if col not in df.columns:
            print(f"   Warning: column '{col}' missing in {year}")
            return pd.DataFrame()

    # Filter to US‑CHN dyad (both directions)
    mask_us_chn = (
        ((df['Actor1Code'] == US_CODE) & (df['Actor2Code'] == CHN_CODE)) |
        ((df['Actor1Code'] == CHN_CODE) & (df['Actor2Code'] == US_CODE))
    )
    df_dyad = df.loc[mask_us_chn, needed].copy()
    if df_dyad.empty:
        return pd.DataFrame()

    df_dyad.columns = ['date', 'Actor1Code', 'Actor2Code', 'EventRootCode', 'GoldsteinScale']
    df_dyad['date'] = pd.to_datetime(df_dyad['date'], format='%Y%m%d', errors='coerce')
    df_dyad['EventRootCode'] = pd.to_numeric(df_dyad['EventRootCode'], errors='coerce')
    df_dyad['GoldsteinScale'] = pd.to_numeric(df_dyad['GoldsteinScale'], errors='coerce')
    df_dyad = df_dyad.dropna(subset=['EventRootCode', 'GoldsteinScale'])

    # Binary indicators
    df_dyad['protest']    = df_dyad['EventRootCode'].isin(PROTEST_CODES).astype(int)
    df_dyad['military']   = df_dyad['EventRootCode'].isin(MILITARY_CODES).astype(int)
    df_dyad['diplomatic'] = df_dyad['EventRootCode'].isin(DIPLOMATIC_CODES).astype(int)
    df_dyad['root_class'] = ((df_dyad['EventRootCode'] - 1) // 4 + 1).astype(int)

    return df_dyad

# ── Fetch all years ─────────────────────────────────────────────────────────────
all_years = []
for yr in range(1990, 2023):
    cache_file = cache_dir / f"uschn_events_{yr}.parquet"
    if cache_file.exists():
        df_yr = pd.read_parquet(cache_file)
        print(f"{yr}: {len(df_yr)} events (cached)")
    else:
        print(f"{yr}: fetching...", end=" ", flush=True)
        df_yr = fetch_year_v1(yr)
        print(f"{len(df_yr)} events")
        if not df_yr.empty:
            df_yr.to_parquet(cache_file)
        time.sleep(0.1)
    if not df_yr.empty:
        all_years.append(df_yr)

if all_years:
    df_all_v1 = pd.concat(all_years, ignore_index=True)
    df_all_v1.to_parquet(out_dir / "gdel_uschn_events_raw_v1.parquet")
    print(f"\nTotal US‑CHN events (v1): {len(df_all_v1)}")
else:
    df_all_v1 = pd.DataFrame()
    print("No events found.")

here
1990: 918 events (cached)
1991: 1798 events (cached)
1992: 1416 events (cached)
1993: 1569 events (cached)
1994: 3027 events (cached)
1995: 5717 events (cached)
1996: 7476 events (cached)
1997: 10868 events (cached)
1998: 10391 events (cached)
1999: 13866 events (cached)
2000: 7452 events (cached)
2001: 13245 events (cached)
2002: 6310 events (cached)
2003: 11125 events (cached)
2004: 7855 events (cached)
2005: 5977 events (cached)
2006: 14741 events (cached)
2007: 25903 events (cached)
2008: 28107 events (cached)
2009: fetching...    Error 2009: Unable to allocate 179. MiB for an array with shape (1, 23464598) and data type int64
0 events
2010: fetching...    Error 2010: Unable to allocate 687. MiB for an array with shape (4, 22502301) and data type float64
0 events
2011: fetching... 

: 

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

out_dir = Path.cwd().parent / "data" / "03_nlp"
cache_dir = out_dir / "gdel_yearly_cache_v1"
cache_dir.mkdir(exist_ok=True)

from gdelt import gdelt
gd1 = gdelt(version=1)

US_CODE  = "USA"
CHN_CODE = "CHN"

PROTEST_CODES    = {14, 15}
MILITARY_CODES   = {19, 20}
DIPLOMATIC_CODES = {3, 4, 5, 6}

def fetch_interval(start, end):
    """
    Fetch events for a short interval (month) using coverage=True.
    Returns a cleaned US‑CHN DataFrame or None.
    """
    try:
        # coverage=True ensures all daily files are downloaded
        df = gd1.Search([start, end],
                         table='events',
                         output='pd',
                         coverage=True)
    except Exception as e:
        print(f"   Error: {e}")
        return None

    if df is None or df.empty:
        return None

    needed = ['SQLDATE', 'Actor1Code', 'Actor2Code', 'EventRootCode', 'GoldsteinScale']
    for col in needed:
        if col not in df.columns:
            return None

    # Filter to US‑CHN dyad (both directions)
    mask = (
        ((df['Actor1Code'] == US_CODE) & (df['Actor2Code'] == CHN_CODE)) |
        ((df['Actor1Code'] == CHN_CODE) & (df['Actor2Code'] == US_CODE))
    )
    df_dyad = df.loc[mask, needed].copy()
    if df_dyad.empty:
        return None

    df_dyad.columns = ['date', 'Actor1Code', 'Actor2Code', 'EventRootCode', 'GoldsteinScale']
    df_dyad['date'] = pd.to_datetime(df_dyad['date'], format='%Y%m%d', errors='coerce')
    df_dyad['EventRootCode'] = pd.to_numeric(df_dyad['EventRootCode'], errors='coerce')
    df_dyad['GoldsteinScale'] = pd.to_numeric(df_dyad['GoldsteinScale'], errors='coerce')
    df_dyad = df_dyad.dropna(subset=['EventRootCode', 'GoldsteinScale'])

    df_dyad['protest']    = df_dyad['EventRootCode'].isin(PROTEST_CODES).astype(int)
    df_dyad['military']   = df_dyad['EventRootCode'].isin(MILITARY_CODES).astype(int)
    df_dyad['diplomatic'] = df_dyad['EventRootCode'].isin(DIPLOMATIC_CODES).astype(int)
    df_dyad['root_class'] = ((df_dyad['EventRootCode'] - 1) // 4 + 1).astype(int)
    return df_dyad

# ── Process years ───────────────────────────────────────────────────────────────
all_years_data = []

for yr in range(1990, 2023):
    cache_file = cache_dir / f"uschn_events_{yr}.parquet"
    if cache_file.exists():
        df_yr = pd.read_parquet(cache_file)
        print(f"{yr}: {len(df_yr)} events (cached)")
        all_years_data.append(df_yr)
        continue

    # If not cached and year >= 2009 → fetch month‑by‑month
    if yr >= 2009:
        print(f"{yr}: fetching month‑by‑month...")
        monthly_parts = []
        for month in range(1, 13):
            start = f"{yr}-{month:02d}-01"
            # Compute end date: last day of month
            if month == 12:
                end = f"{yr}-12-31"
            else:
                next_month = month + 1
                end = f"{yr}-{next_month:02d}-01"
                end = pd.Timestamp(end) - pd.Timedelta(days=1)
                end = end.strftime("%Y-%m-%d")
            df_month = fetch_interval(start, end)
            if df_month is not None and not df_month.empty:
                monthly_parts.append(df_month)
            # tiny pause
            time.sleep(0.05)
        if monthly_parts:
            df_yr = pd.concat(monthly_parts, ignore_index=True)
            df_yr.to_parquet(cache_file)
            print(f"  → saved {len(df_yr)} events for {yr}")
        else:
            df_yr = pd.DataFrame()
            print(f"  → 0 events for {yr}")
    else:
        # For pre‑2009, fallback to original yearly fetch (shouldn't need it if cached)
        print(f"{yr}: fetching year...", end=" ")
        df_full = fetch_interval(f"{yr}-01-01", f"{yr}-12-31")
        if df_full is not None and not df_full.empty:
            df_yr = df_full
            df_yr.to_parquet(cache_file)
            print(f"{len(df_yr)} events")
        else:
            df_yr = pd.DataFrame()
            print("0 events")

    if not df_yr.empty:
        all_years_data.append(df_yr)
    time.sleep(0.1)

# ── Combine and save ──────────────────────────────────────────────────────────
if all_years_data:
    df_all_v1 = pd.concat(all_years_data, ignore_index=True)
    df_all_v1.to_parquet(out_dir / "gdel_uschn_events_raw_v1.parquet")
    print(f"\nTotal US‑CHN events (v1): {len(df_all_v1)}")
else:
    df_all_v1 = pd.DataFrame()
    print("No events found.")

1990: 918 events (cached)
1991: 1798 events (cached)
1992: 1416 events (cached)
1993: 1569 events (cached)
1994: 3027 events (cached)
1995: 5717 events (cached)
1996: 7476 events (cached)
1997: 10868 events (cached)
1998: 10391 events (cached)
1999: 13866 events (cached)
2000: 7452 events (cached)
2001: 13245 events (cached)
2002: 6310 events (cached)
2003: 11125 events (cached)
2004: 7855 events (cached)
2005: 5977 events (cached)
2006: 14741 events (cached)
2007: 25903 events (cached)
2008: 28107 events (cached)
2009: 56734 events (cached)
2010: 59582 events (cached)
2011: 62587 events (cached)
2012: 76444 events (cached)
2013: 35339 events (cached)
2014: 102776 events (cached)
2015: 130584 events (cached)
2016: 152338 events (cached)
2017: 27453 events (cached)
2018: fetching month‑by‑month...
  → saved 168744 events for 2018
2019: fetching month‑by‑month...
   Error: None: None
   Error: None: None
  → saved 138153 events for 2019
2020: fetching month‑by‑month...
   Error: None: No

### GDELT Data Verification

**Why:** Confirm that the downloaded US‑CHN event data covers every month from 1990‑01 to 2022‑02, that no days are missing, and that yearly/monthly counts are plausible.  
**Method:**  
- Load the full raw Parquet file.  
- Check that every expected month appears in the resampled counts.  
- Identify months with zero events (potential download failures).  
- Plot the monthly total event count so you can visually spot gaps or anomalies.  
- Print yearly totals for manual cross‑check with known historical patterns (e.g., 9/11 spike, 2015‑2016 tension, etc.).

In [4]:
import pandas as pd
from pathlib import Path

out_dir = Path.cwd().parent / "data" / "03_nlp"
raw_path = out_dir / "gdel_uschn_events_raw_v1.parquet"

# Load & clip to sample window
df = pd.read_parquet(raw_path)
df['date'] = pd.to_datetime(df['date'])
mask = (df['date'] >= '1990-01-01') & (df['date'] <= '2022-02-28')
df_sample = df.loc[mask].copy()

print(f"Events in window: {len(df_sample)} (out of {len(df)} total)")

# Monthly aggregation (like your final CSV)
df_sample.set_index('date', inplace=True)
monthly = df_sample.resample('MS').agg(
    protest_count      = ('protest', 'sum'),
    military_count     = ('military', 'sum'),
    diplomatic_count   = ('diplomatic', 'sum'),
    total_events       = ('protest', 'size'),  # count of rows
    goldstein_mean     = ('GoldsteinScale', 'mean'),
    goldstein_sum      = ('GoldsteinScale', 'sum')
)

# Align to 386-month index
full_index = pd.date_range('1990-01-01', '2022-02-01', freq='MS')
monthly = monthly.reindex(full_index)

# Percentage of months with zero total events
zero_event_months = monthly['total_events'].isna() | (monthly['total_events'] == 0)
pct_missing = zero_event_months.mean() * 100
print(f"\nMonths with zero events: {zero_event_months.sum()} out of {len(full_index)} ({pct_missing:.1f}%)")

# If any, show them
if zero_event_months.any():
    print("Missing months:")
    for m in monthly.index[zero_event_months]:
        print(f"  {m.strftime('%Y-%m')}")

# Show months with suspiciously low counts (less than 10% of median monthly events)
median_events = monthly['total_events'].median()
low_threshold = max(10, median_events * 0.1)
low_months = (monthly['total_events'] < low_threshold) & ~zero_event_months
if low_months.any():
    print(f"\n⚠ Suspiciously low months (<{low_threshold:.0f} events):")
    for m in monthly.index[low_months]:
        print(f"  {m.strftime('%Y-%m')}: {monthly.loc[m, 'total_events']:.0f} events")

# Yearly sums
yearly = monthly['total_events'].resample('YS').sum()
print("\nYearly totals (sample window only):")
for yr, val in yearly.items():
    print(f"  {yr.year}: {int(val)}")

Events in window: 1448678 (out of 1513674 total)

Months with zero events: 0 out of 386 (0.0%)

⚠ Suspiciously low months (<102 events):
  1990-01: 70 events
  1990-02: 86 events
  1990-03: 78 events
  1990-04: 60 events
  1990-06: 101 events
  1990-07: 87 events
  1990-08: 48 events
  1990-09: 39 events
  1990-10: 45 events
  1990-12: 87 events
  1991-01: 40 events
  1991-02: 22 events
  1992-02: 101 events
  1992-03: 44 events
  1992-04: 76 events
  1992-06: 93 events
  1992-07: 58 events
  1992-08: 64 events
  1992-10: 89 events
  1993-01: 79 events
  1993-02: 49 events
  1993-03: 85 events
  1993-04: 61 events
  1993-06: 77 events
  2013-07: 96 events
  2020-08: 44 events

Yearly totals (sample window only):
  1990: 918
  1991: 1798
  1992: 1416
  1993: 1569
  1994: 3027
  1995: 5717
  1996: 7476
  1997: 10868
  1998: 10391
  1999: 13866
  2000: 7452
  2001: 13245
  2002: 6310
  2003: 11136
  2004: 7895
  2005: 6032
  2006: 14801
  2007: 25916
  2008: 28218
  2009: 56787
  2010: 59

In [9]:
import pandas as pd
from pathlib import Path

out_dir = Path.cwd().parent / "data" / "03_nlp"

# Load raw data
df = pd.read_parquet(out_dir / "gdel_uschn_events_raw_v1.parquet")
df['date'] = pd.to_datetime(df['date'])
df = df[(df['date'] >= '1990-01-01') & (df['date'] <= '2022-02-28')].copy()
df.set_index('date', inplace=True)

# ── Monthly aggregation ──────────────────────────────────────────────────────
monthly = df.resample('MS').agg(
    protest_count      = ('protest', 'sum'),
    military_count     = ('military', 'sum'),
    diplomatic_count   = ('diplomatic', 'sum'),
    total_events       = ('protest', 'size'),
    goldstein_mean     = ('GoldsteinScale', 'mean'),
    goldstein_sum      = ('GoldsteinScale', 'sum')
)

# ── THEME MATRIX: one column per EventRootCode (1‑20) ─────────────────────────
# Create dummies from the original EventRootCode column
theme_dummies = pd.get_dummies(df['EventRootCode'], prefix='theme')
# Ensure all 20 columns exist (fill missing ones with 0)
expected_cols = [f'theme_{c}' for c in range(1, 21)]
theme_dummies = theme_dummies.reindex(columns=expected_cols, fill_value=0)
# Aggregate to monthly sums
theme_monthly = theme_dummies.resample('MS').sum()

# ── Combine and align ────────────────────────────────────────────────────────
full_idx = pd.date_range('1990-01-01', '2022-02-01', freq='MS')
df_monthly = pd.concat([monthly, theme_monthly], axis=1).reindex(full_idx)

# Save
clean_csv = out_dir / "gdel_events_monthly_clean.csv"
df_monthly.to_csv(clean_csv)
print(f"Saved final monthly CSV → {clean_csv}")

# Verification
print(f"Shape: {df_monthly.shape}")
print(f"theme_11 sum (demand, should be > 0): {df_monthly['theme_11'].sum():.0f}")
print(f"theme_19 sum (fight, should be > 0): {df_monthly['theme_19'].sum():.0f}")
print(f"theme_20 sum (extreme, should be > 0): {df_monthly['theme_20'].sum():.0f}")
print(f"Columns with any data: {df_monthly.fillna(0).sum().gt(0).sum()}")

Saved final monthly CSV → c:\Users\HP\Desktop\replication+contribution\data\03_nlp\gdel_events_monthly_clean.csv
Shape: (386, 26)
theme_11 sum (demand, should be > 0): 119564
theme_19 sum (fight, should be > 0): 44957
theme_20 sum (extreme, should be > 0): 267
Columns with any data: 26


In [10]:
import pandas as pd
from pathlib import Path

out_dir = Path.cwd().parent / "data" / "03_nlp"
df = pd.read_csv(out_dir / "gdel_events_monthly_clean.csv", index_col=0, parse_dates=True)

# Show a month where we know there were many material conflict events (e.g. 2016-12)
print("December 2016 (peak tension):")
print(df.loc['2016-12', ['theme_11','theme_12','theme_13','theme_14','theme_15','theme_16','theme_17','theme_18','theme_19','theme_20']])
print()

# Show sums per theme column (should match the diagnostic from raw data)
theme_cols = [c for c in df.columns if c.startswith('theme_')]
print("Total counts per theme column (full sample):")
print(df[theme_cols].sum().astype(int))

December 2016 (peak tension):
            theme_11  theme_12  theme_13  theme_14  theme_15  theme_16  \
2016-12-01      1529       472       426       158        29       173   

            theme_17  theme_18  theme_19  theme_20  
2016-12-01       355        93       467         0  

Total counts per theme column (full sample):
theme_1     171605
theme_2      85404
theme_3     133019
theme_4     425043
theme_5     128789
theme_6      78197
theme_7      33422
theme_8      33659
theme_9      20812
theme_10     20347
theme_11    119564
theme_12     35774
theme_13     31249
theme_14      6807
theme_15      4407
theme_16     23962
theme_17     45247
theme_18      6147
theme_19     44957
theme_20       267
dtype: int64
